# books_pub_review 전처리 및 적재

- 데이터: `data/processed/books_with_summarized_review.csv`
- 임베딩 컬럼: `pub_review_summarized`
- 참고: `book_sample_preprocessed_v3.ipynb`, `summarized_review_analysis_001.ipynb`

In [13]:
import re
import math
import pandas as pd


## 1. 데이터 로드

In [14]:
pd.set_option('display.max_colwidth', 60)

df = pd.read_csv('../../data/processed/books_with_summarized_review.csv')
print(f'레코드 수: {len(df):,}')
print(f'컬럼: {df.columns.tolist()}')
df.head(3)

레코드 수: 47,787
컬럼: ['isbn', 'title', 'author', 'publisher', 'publish_date', 'page', 'book_intro', 'pub_review', 'cate_depth1', 'ori_cover_s', 'aladin_category_id', 'aladin_category_name', 'pub_review_summarized']


,isbn,title,author,publisher,publish_date,page,book_intro,pub_review,cate_depth1,ori_cover_s,aladin_category_id,aladin_category_name,pub_review_summarized
0,9788987835730,직장인을 위한 주말 가족 산행기 100선 2,"이상훈, 고광문, 이은빈, 이은찬 공저",화담,2014-03-24,304,아빠와 엄마 그리고 두 아이들의 산행기는 계속된다. 여전히 주말이면 산을 찾아 가족은 떠날 채비를 한...,산에 왜 갔지 ? 그 두 번째 이야기 준비 걸음 아빠 노릇 제대로 못했으니 봉사 한 번 해 주리라는 ...,"['여행', '건강/취미']",https://image.aladin.co.kr/product/3917/39/cover500/8987...,53529.0,국내도서>건강/취미>등산/캠핑,- 저자가 아이들과 함께 시작한 등산 여정을 담은 에세이임.\n\n- '첫 걸음'에서 백 개의 산행을...
1,9788963717715,부요황후 1,천하귀원 저/김지혜 역,파란썸 (파란미디어),2020-07-14,420,"중국 누적 조회 수 100억 돌파, 드라마 부요황후 원작 소설 유머 넘치는 문장과 탄탄한 이야기, 끊...","당신을 위해서라면 번잡한 세상 풍파, 그 어떠한 고난도 두렵지 않다! 중국 누적 조회 수 100억 돌...",['소설'],https://image.aladin.co.kr/product/24410/85/cover500/896...,51126.0,국내도서>소설/시/희곡>로맨스소설>외국 로맨스소설,"부요황후는 천하귀원 작가의 소설로, 2012년 진강시문예상과 전국우수여성문학상을 수상했으며 2018..."
2,9791185860084,"시장이 두근두근 2 - 대전, 대구, 광주, 부산, 제주",이희준 저,이야기나무,2015-07-07,360,"서울부터 수원, 인천, 강원, 대전, 대구, 광주, 부산, 그리고 제주도까지 전국에 흩어진 435곳의...",전국에 흩어진 전통시장을 20대 청춘의 시선으로 기록하다! 공모전 준비와 스펙 쌓기에 바쁜 20대 청...,['여행'],https://image.aladin.co.kr/product/6266/95/cover500/k982...,50846.0,국내도서>여행>국내 여행가이드>전국여행 가이드북,"- 20대 청춘이 전통시장을 방문하면서 느낀 점과 그 과정을 기록한 책이다. \n- 전국에 1,372..."


## 2. pub_review_summarized 전처리 (줄바꿈·특수문자 정리)

In [15]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.replace('\n', ' ').replace('\r', ' ')   # 줄바꿈 제거
    text = re.sub(r'[\u200b\u00a0\t]+', ' ', text)      # 제로폭 공백, non-breaking space, 탭
    text = re.sub(r'\s{2,}', ' ', text)                  # 연속 공백 단일화
    return text.strip()

df['pub_review_summarized'] = df['pub_review_summarized'].apply(clean_text)

# 전처리 후 길이 재계산
df['summ_len'] = df['pub_review_summarized'].str.len()

print('전처리 후 pub_review_summarized 길이 통계:')
print(df['summ_len'].describe(percentiles=[.25, .5, .75, .95]).round(0))

전처리 후 pub_review_summarized 길이 통계:
count    47787.0
mean       365.0
std        113.0
min         29.0
25%        284.0
50%        353.0
75%        432.0
95%        586.0
max        776.0
Name: summ_len, dtype: float64


## 3. 이상 케이스 필터링

`summarized_review_analysis_001.ipynb` 분석 결과 기반으로 아래 케이스 제외:
- 요약이 원본보다 긴 케이스 (`summ_len / orig_len > 2`)
- 요약 100자 미만 (50자 미만 포함)
- "서평 없음" 패턴 포함

In [16]:
before = len(df)

df['orig_len'] = df['pub_review'].str.len()
df['len_ratio'] = df['summ_len'] / df['orig_len']

mask_hallucination = df['len_ratio'] > 2
mask_short         = df['summ_len'] < 100
mask_no_review     = df['pub_review_summarized'].str.contains(
    '서평 없음|서평 정보|제공되지 않', na=False
)

print('=== 필터링 전 ===')
print(f'총 레코드: {before:,}건')
print()
print('=== 제외 케이스별 건수 ===')
print(f'요약이 원본의 2배 초과 (할루시네이션 의심): {mask_hallucination.sum():>6,}건 ({mask_hallucination.sum()/before*100:.2f}%)')
print(f'요약 100자 미만:                          {mask_short.sum():>6,}건 ({mask_short.sum()/before*100:.2f}%)')
print(f'"서평 없음" 패턴 포함:                     {mask_no_review.sum():>6,}건 ({mask_no_review.sum()/before*100:.2f}%)')

mask_any = mask_hallucination | mask_short | mask_no_review
print(f'\n중복 제거 후 실제 제외 건수: {mask_any.sum():,}건 ({mask_any.sum()/before*100:.2f}%)')

df = df[~mask_any].reset_index(drop=True)

after = len(df)
print(f'\n=== 필터링 후 ===')
print(f'남은 레코드: {after:,}건 (제거: {before - after:,}건, {(before - after)/before*100:.2f}%)')

=== 필터링 전 ===
총 레코드: 47,787건

=== 제외 케이스별 건수 ===
요약이 원본의 2배 초과 (할루시네이션 의심):  1,094건 (2.29%)
요약 100자 미만:                              24건 (0.05%)
"서평 없음" 패턴 포함:                         12건 (0.03%)

중복 제거 후 실제 제외 건수: 1,118건 (2.34%)

=== 필터링 후 ===
남은 레코드: 46,669건 (제거: 1,118건, 2.34%)


## 4. 카테고리 분리 (aladin_category_name → 4개 컬럼)

In [17]:
category_split = df['aladin_category_name'].str.split('>', expand=True)

df['category_origin'] = category_split[0].str.strip()
df['category_large']  = category_split[1].str.strip() if 1 in category_split.columns else None
df['category_medium'] = category_split[2].str.strip() if 2 in category_split.columns else None
df['category_small']  = category_split[3].str.strip() if 3 in category_split.columns else None

print('국내/외국 구분:', df['category_origin'].value_counts().to_dict())
print('대분류 종류:', df['category_large'].nunique())
print('중분류 결측:', df['category_medium'].isna().sum())
print('소분류 결측:', df['category_small'].isna().sum())

국내/외국 구분: {'국내도서': 46615, '외국도서': 1, '전자책': 1}
대분류 종류: 29
중분류 결측: 54
소분류 결측: 7738


## 5. author 컬럼 파싱 (저/역/감수 분리)

In [18]:
def parse_author(text):
    if not isinstance(text, str):
        return [], [], []

    author, translator, supervisor = [], [], []

    text = re.sub(r'\s*외\s*\d+명', '', text)
    text = re.sub(r'글,\s*그림', '글그림', text)
    text = re.sub(r'글,\s*기획', '글기획', text)
    text = re.sub(r'글,\s*만화', '글만화', text)
    text = re.sub(r'기획,\s*제작', '기획제작', text)

    role_pattern = r'(공저|편저|편역|공역|저|역|감수|자문|글그림|글기획|글만화|기획제작|글|그림|만화|기획|제작|편)$'

    for seg in text.split('/'):
        last_name = None
        for part in [p.strip() for p in seg.split(',') if p.strip()]:
            tag_match = re.search(role_pattern, part)
            if tag_match:
                tag  = tag_match.group()
                name = part[:tag_match.start()].strip() or last_name
                if not name:
                    continue
                last_name = name
                if tag in ('역', '공역', '편역'):
                    translator.append(name)
                elif tag in ('감수', '자문'):
                    supervisor.append(name)
                else:
                    author.append(name)
            else:
                author.append(part)
                last_name = part

    return author, translator, supervisor

df['ori_author'] = df['author']
df[['author', 'translator', 'supervisor']] = df['author'].apply(
    lambda x: pd.Series(parse_author(x))
)

print(df[['ori_author', 'author', 'translator', 'supervisor']].head(5))

                      ori_author                author translator supervisor
0          이상훈, 고광문, 이은빈, 이은찬 공저  [이상훈, 고광문, 이은빈, 이은찬]         []         []
1                   천하귀원 저/김지혜 역                [천하귀원]      [김지혜]         []
2                          이희준 저                 [이희준]         []         []
3  중국사천대학 저 / 이수진 역 / 멘사수학연구소 감수              [중국사천대학]      [이수진]  [멘사수학연구소]
4                          편집부 저                 [편집부]         []         []


## 6. publish_date 처리 (timezone 추가)

In [19]:
df['publish_date_dt'] = pd.to_datetime(df['publish_date'], errors='coerce').dt.tz_localize(
    'Asia/Seoul', nonexistent='NaT', ambiguous='NaT'
)
df['publish_year']    = df['publish_date_dt'].dt.year
df['publish_date_dt'] = df['publish_date_dt'].dt.strftime('%Y-%m-%dT%H:%M:%S+09:00')

print(df['publish_year'].value_counts().sort_index())

publish_year
1974.0       1
1988.0       1
1990.0       2
1991.0       2
1992.0       4
1993.0       3
1994.0      13
1995.0      17
1996.0      15
1997.0      31
1998.0      25
1999.0      83
2000.0     238
2001.0     281
2002.0     235
2003.0     262
2004.0     277
2005.0     373
2006.0     597
2007.0     899
2008.0    1480
2009.0    1719
2010.0    1831
2011.0    1818
2012.0    1907
2013.0    2143
2014.0    2296
2015.0    2541
2016.0    2457
2017.0    2312
2018.0    2419
2019.0    2837
2020.0    2973
2021.0    3245
2022.0    3134
2023.0    3028
2024.0    3246
2025.0    1919
2026.0       4
Name: count, dtype: int64


## 7. 컬럼 정리 및 최종 확인

In [20]:
cols = [
    'isbn', 'title', 'author', 'translator',
    'supervisor', 'publisher', 'publish_year',
    'publish_date_dt', 'page', 'book_intro',
    'pub_review', 'pub_review_summarized',
    'cate_depth1', 'ori_cover_s', 'aladin_category_name',
    'aladin_category_id', 'category_origin',
    'category_large', 'category_medium', 'category_small',
]
df = df[cols].reset_index(drop=True)

print(f'최종 레코드 수: {len(df):,}')
print(f'컬럼 수: {len(df.columns)}')
df.head(3)

최종 레코드 수: 46,669
컬럼 수: 20


,isbn,title,author,translator,supervisor,publisher,publish_year,publish_date_dt,page,book_intro,pub_review,pub_review_summarized,cate_depth1,ori_cover_s,aladin_category_name,aladin_category_id,category_origin,category_large,category_medium,category_small
0,9788987835730,직장인을 위한 주말 가족 산행기 100선 2,"[이상훈, 고광문, 이은빈, 이은찬]",[],[],화담,2014.0,2014-03-24T00:00:00+09:00,304,아빠와 엄마 그리고 두 아이들의 산행기는 계속된다. 여전히 주말이면 산을 찾아 가족은 떠날 채비를 한...,산에 왜 갔지 ? 그 두 번째 이야기 준비 걸음 아빠 노릇 제대로 못했으니 봉사 한 번 해 주리라는 ...,- 저자가 아이들과 함께 시작한 등산 여정을 담은 에세이임. - '첫 걸음'에서 백 개의 산행을 목표...,"['여행', '건강/취미']",https://image.aladin.co.kr/product/3917/39/cover500/8987...,국내도서>건강/취미>등산/캠핑,53529.0,국내도서,건강/취미,등산/캠핑,NaN
1,9788963717715,부요황후 1,[천하귀원],[김지혜],[],파란썸 (파란미디어),2020.0,2020-07-14T00:00:00+09:00,420,"중국 누적 조회 수 100억 돌파, 드라마 부요황후 원작 소설 유머 넘치는 문장과 탄탄한 이야기, 끊...","당신을 위해서라면 번잡한 세상 풍파, 그 어떠한 고난도 두렵지 않다! 중국 누적 조회 수 100억 돌...","부요황후는 천하귀원 작가의 소설로, 2012년 진강시문예상과 전국우수여성문학상을 수상했으며 2018년...",['소설'],https://image.aladin.co.kr/product/24410/85/cover500/896...,국내도서>소설/시/희곡>로맨스소설>외국 로맨스소설,51126.0,국내도서,소설/시/희곡,로맨스소설,외국 로맨스소설
2,9791185860084,"시장이 두근두근 2 - 대전, 대구, 광주, 부산, 제주",[이희준],[],[],이야기나무,2015.0,2015-07-07T00:00:00+09:00,360,"서울부터 수원, 인천, 강원, 대전, 대구, 광주, 부산, 그리고 제주도까지 전국에 흩어진 435곳의...",전국에 흩어진 전통시장을 20대 청춘의 시선으로 기록하다! 공모전 준비와 스펙 쌓기에 바쁜 20대 청...,"- 20대 청춘이 전통시장을 방문하면서 느낀 점과 그 과정을 기록한 책이다. - 전국에 1,372개의...",['여행'],https://image.aladin.co.kr/product/6266/95/cover500/k982...,국내도서>여행>국내 여행가이드>전국여행 가이드북,50846.0,국내도서,여행,국내 여행가이드,전국여행 가이드북


## 8. 데이터 적재

In [21]:
import sys
import os
import io
import uuid
import contextlib
from tqdm import tqdm
from qdrant_client.models import PointStruct

sys.path.insert(0, os.path.abspath('../..'))

from src.db.qdrant import QdrantDB
from src.embedding.embedder import LocalEmbedder

In [22]:
embedder = LocalEmbedder()
db = QdrantDB(vector_size=1024, timeout=1200)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10008.99it/s]


In [23]:
# 컬렉션 이름: books_pub_review_{N}k  (N = 천 단위)
N = len(df) // 1000
COLLECTION = f'books_pub_review_{N}k'
BATCH_SIZE  = 64

INDEXES = [
    ('cate_depth1',     'keyword'),
    ('publish_year',    'integer'),
    ('author',          'text'),
    ('title',           'text'),
    ('category_origin', 'keyword'),
    ('category_large',  'keyword'),
    ('category_medium', 'keyword'),
    ('category_small',  'keyword'),
]

print(f'컬렉션: {COLLECTION}')
print(f'적재 건수: {len(df):,}')

컬렉션: books_pub_review_46k
적재 건수: 46,669


In [24]:
# 기존 컬렉션 목록 확인
collections = db.get_collections().collections
for c in collections:
    info = db.client.get_collection(c.name)
    print(f"{c.name} — 벡터 수: {info.points_count:,}, 차원: {info.config.params.vectors.size}")

if not collections:
    print('저장된 collection이 없습니다.')

books_intro_48k — 벡터 수: 47,308, 차원: 1024
books_merged_48k — 벡터 수: 47,308, 차원: 1024


In [26]:
def to_val(v):
    if isinstance(v, float) and math.isnan(v):
        return None
    return v

def to_int(v):
    if isinstance(v, float) and math.isnan(v):
        return None
    return int(v)

existing = [c.name for c in db.get_collections().collections]
if COLLECTION not in existing:
    db.create_collection(COLLECTION)
    for field, schema in INDEXES:
        db.create_payload_index(COLLECTION, field, schema)
else:
    print(f'컬렉션 이미 존재: {COLLECTION}')

records = df.to_dict('records')

for i in tqdm(range(0, len(records), BATCH_SIZE), desc=f'{COLLECTION} 적재'):
    batch = records[i:i + BATCH_SIZE]
    vectors = embedder.embed_batch([r['pub_review_summarized'] for r in batch])

    points = [
        PointStruct(
            id=str(uuid.uuid5(uuid.NAMESPACE_DNS, str(row['isbn']))),
            vector=vectors[j],
            payload={
                'isbn':                 row['isbn'],
                'title':                to_val(row['title']),
                'author':               row['author'],
                'translator':           row['translator'],
                'supervisor':           row['supervisor'],
                'publisher':            to_val(row['publisher']),
                'publish_year':         to_int(row['publish_year']),
                'publish_date':         to_val(row['publish_date_dt']),
                'page':                 to_int(row['page']),
                'book_intro':           to_val(row['book_intro']),
                'pub_review':           to_val(row['pub_review']),
                'pub_review_summarized': to_val(row['pub_review_summarized']),
                'cate_depth1':          to_val(row['cate_depth1']),
                'cover_url':            to_val(row['ori_cover_s']),
                'aladin_category_id':   to_int(row['aladin_category_id']),
                'aladin_category_name': to_val(row['aladin_category_name']),
                'category_origin':      to_val(row['category_origin']),
                'category_large':       to_val(row['category_large']),
                'category_medium':      to_val(row['category_medium']),
                'category_small':       to_val(row['category_small']),
            }
        )
        for j, row in enumerate(batch)
    ]

    with contextlib.redirect_stdout(io.StringIO()):
        db.insert(COLLECTION, points)

print(f'\n총 {len(records):,}건 적재 완료 → {COLLECTION}')

컬렉션 이미 존재: books_pub_review_46k


books_pub_review_46k 적재: 100%|██████████| 730/730 [1:07:45<00:00,  5.57s/it]


총 46,669건 적재 완료 → books_pub_review_46k


In [27]:
# 적재 결과 확인
info = db.client.get_collection(COLLECTION)
print(f'컬렉션: {COLLECTION}')
print(f'벡터 수: {info.points_count:,}')
print(f'벡터 차원: {info.config.params.vectors.size}')

컬렉션: books_pub_review_46k
벡터 수: 46,669
벡터 차원: 1024
